# Laboratorio 5 — Navegación Basada en Comportamientos

Jesús Manuel Aragón Buitrago

Juan Carlos González Ibarra

Ángel Rivera Amórtegui

Duvan Stiven Tique Osorio

Universidad Nacional de Colombia — Facultad de Ingeniería

Laboratorio de Sistemas Inteligentes en Robótica (Labsir)

Curso: Fundamentos de Robótica Móvil

## Introducción

En esta práctica se aborda la navegación de robots móviles desde dos paradigmas complementarios: la navegación deliberativa, basada en modelos internos del entorno y planeación explícita de trayectorias, y la navegación reactiva, en la que el robot responde directamente a los estímulos de sus sensores sin necesidad de un mapa previo. Los sistemas híbridos, que combinan ambos enfoques, ofrecen una solución más robusta y flexible para enfrentar los desafíos de la navegación autónoma. En este laboratorio se implementan y ponen a prueba dos algoritmos basados en comportamientos sobre el robot LEGO EV3: un algoritmo tipo Bug para evadir obstáculos entre dos puntos marcados en el piso, y un algoritmo de seguimiento de pared para resolver un laberinto. Ambas soluciones se documentan con pseudocódigo, diagramas, código fuente y registro audiovisual de su ejecución.

## Investigación preliminar

### Navegación planeada (deliberativa) vs. navegación basada en comportamientos (reactiva)

**Navegación deliberativa:**
- Depende de un modelo o mapa interno del entorno (construido previamente o mediante SLAM) sobre el cual se planea la ruta completa antes de moverse.
- Al calcular explícitamente la trayectoria, tiende a producir movimientos más "óptimos" o predecibles, pero es sensible a errores de modelado y a cambios dinámicos del entorno no contemplados en el mapa.

**Navegación reactiva (basada en comportamientos):**
- No requiere un modelo global del entorno; cada comportamiento reacciona directamente a la lectura instantánea de los sensores (contacto, distancia, color, etc.), acoplando percepción y acción.
- Responde con baja latencia a cambios del entorno no anticipados (obstáculos dinámicos), a costa de trayectorias menos eficientes y de la posibilidad de quedar atrapada en mínimos locales si los comportamientos no están bien combinados.

### Rodney Brooks

[PENDIENTE — completar en máx. 2 párrafos: arquitectura de subsunción (subsumption architecture), los robots Genghis y Allen del MIT, y la idea de que "el mundo es su propio mejor modelo" como alternativa a la robótica basada en representación simbólica.]

### Mark Tilden

[PENDIENTE — completar en máx. 2 párrafos: robótica BEAM (Biology, Electronics, Aesthetics, Mechanics), robots solares tipo "Solarroller"/robots caminantes analógicos sin microprocesador, control mediante redes nerviosas analógicas.]

### Algoritmos de planeación de rutas en espacios con obstáculos

1. **Campos de potencial artificiales (Potential Fields):** la meta genera una fuerza atractiva y los obstáculos fuerzas repulsivas; el robot sigue el gradiente resultante de ambas.
2. **Grafos de visibilidad (Visibility Graphs):** se construye un grafo conectando los vértices de los obstáculos y el robot planea el camino más corto sobre ese grafo.
3. **Descomposición en celdas / búsqueda en rejilla (p. ej. A\*, D\*):** el espacio se discretiza en una cuadrícula y se busca la ruta de menor costo mediante búsqueda heurística.
4. **RRT (Rapidly-exploring Random Trees):** exploración probabilística del espacio de configuraciones mediante árboles aleatorios, útil en espacios de alta dimensión.

### Algoritmos Bug

- **Bug 0:** el robot avanza en línea recta hacia la meta; al toparse con un obstáculo, lo bordea hasta que puede reanudar la línea recta hacia la meta. No garantiza terminar en entornos complejos porque no compara distancias antes de abandonar el borde del obstáculo.
- **Bug 1:** al chocar con un obstáculo, el robot lo recorre por completo, memoriza el punto de menor distancia a la meta sobre ese contorno, y luego vuelve a ese punto para continuar hacia la meta. Garantiza llegar si existe un camino, pero es ineficiente por recorrer el obstáculo entero.
- **Bug 2:** el robot sigue una línea recta imaginaria (m-line) entre el inicio y la meta; al chocar con un obstáculo lo bordea hasta reencontrar esa línea más cerca de la meta que el punto de choque, y ahí la abandona. Suele ser más eficiente que Bug 1.

### Algoritmo de resolución de laberintos

**Seguimiento de pared (wall-following, regla de la mano derecha/izquierda):** el robot mantiene contacto —físico o por sensor de distancia— con una de las paredes del laberinto y la sigue consistentemente en cada intersección (gira hacia la pared seguida cuando hay abertura, sigue recto en pasillo, gira en sentido contrario en callejón sin salida). Si el laberinto es simplemente conexo (sin islas desconectadas de las paredes exteriores), este método garantiza encontrar la salida, aunque no necesariamente por el camino más corto.


## Misión 1: Evadir obstáculos (algoritmo Bug)

**Objetivo:** trasladar el robot EV3 de forma autónoma desde un punto de partida P1 hasta un punto de llegada P2, marcados en el piso con cinta, evadiendo al menos dos obstáculos ubicados sobre la línea que los conecta, con espacio suficiente para que el robot los rodee sin intervención manual.

**Descripción de la solución:** se implementó una variante tipo Bug que combina seguimiento de línea con bordeo de obstáculos. El robot sigue con el sensor de color la cinta que marca la trayectoria P1→P2. Al chocar contra un obstáculo (detectado con el sensor de contacto), retrocede, gira 90° y entra en un comportamiento de bordeo guiado por el sensor ultrasónico, manteniendo una distancia aproximadamente constante al obstáculo mediante un control tipo bang-bang entre los umbrales `DIST_CERCA` y `DIST_LEJOS`. Pasado un tiempo mínimo de bordeo (`TIEMPO_MIN`), el robot comprueba si reencontró la línea con el sensor de color; si es así, retoma el seguimiento de línea. El robot se detiene al detectar el color de meta en P2. Esta solución corresponde a una variante de Bug con contacto físico como disparador de la evasión (similar a Bug 0/Bug 1) y bordeo por distancia en lugar de por contacto continuo.

**Pseudocódigo:**

```
inicializar motores, sensor de contacto, sensor de color, sensor ultrasónico
estado ← SIGUIENDO_LINEA

repetir:
    si estado = SIGUIENDO_LINEA:
        si color_actual = COLOR_META:
            detener motores y terminar
        si_no si sensor_contacto = presionado:
            retroceder brevemente
            girar 90° a la izquierda
            tiempo_inicio_rodeo ← tiempo_actual
            estado ← RODEANDO
        si_no:
            si color_actual = COLOR_LINEA:
                avanzar recto
            si_no:
                corregir trayectoria hacia la línea
    si_no si estado = RODEANDO:
        distancia ← leer sensor ultrasónico
        si distancia > DIST_LEJOS:
            girar hacia el obstáculo (buscarlo)
        si_no si distancia < DIST_CERCA:
            alejarse del obstáculo
        si_no:
            avanzar recto bordeando a distancia constante
        si (tiempo_actual - tiempo_inicio_rodeo) > TIEMPO_MIN y color_actual = COLOR_LINEA:
            detener motores
            estado ← SIGUIENDO_LINEA
    esperar un instante
hasta llegar a la meta
```

**Montaje del espacio de trabajo:**

[PENDIENTE: foto/diagrama con P1, P2, la cinta marcando la trayectoria y la ubicación de los al menos dos obstáculos. Agregar la imagen a `img/` y referenciarla aquí, p. ej. `![](img/bug_montaje.png)`]

**Programa:**


In [ ]:
from time import sleep, time
from ev3dev2.motor import LargeMotor, OUTPUT_B, OUTPUT_C, SpeedPercent
from ev3dev2.sensor.lego import TouchSensor, ColorSensor, UltrasonicSensor

# --- Hardware ---
MotIzq = LargeMotor(OUTPUT_B)
MotDer = LargeMotor(OUTPUT_C)
toque  = TouchSensor('in4')
color  = ColorSensor('in3')
usonic = UltrasonicSensor('in1')

color.mode  = 'COL-COLOR'
usonic.mode = 'US-DIST-CM'

# --- Parámetros ---
VEL_BASE     = 20
DIST_CERCA   = 15    # cm — demasiado cerca del obstáculo
DIST_LEJOS   = 20    # cm — demasiado lejos del obstáculo
TIEMPO_MIN   = 1.5   # segundos antes de buscar línea al rodear

# --- Estados ---
SIGUIENDO_LINEA = 0
RODEANDO        = 1
estado          = SIGUIENDO_LINEA
tiempo_rodeo    = 0


def en_linea():
    return color.color == 4


def seguirLinea():
    if color.color == 4:
        MotIzq.on(SpeedPercent(VEL_BASE))
        MotDer.on(SpeedPercent(VEL_BASE))
    else:
        MotIzq.on(SpeedPercent(0))
        MotDer.on(SpeedPercent(VEL_BASE))


def rodearObstaculo():
    dist = usonic.distance_centimeters
    if dist > DIST_LEJOS:
        MotIzq.on(SpeedPercent(20))    # gira derecha, busca obstáculo
        MotDer.on(SpeedPercent(5))
    elif dist < DIST_CERCA:
        MotIzq.on(SpeedPercent(5))   # gira izquierda, se aleja
        MotDer.on(SpeedPercent(20))
    else:
        MotIzq.on(SpeedPercent(15))   # avanza recto
        MotDer.on(SpeedPercent(15))


def girar90Izquierda():
    MotIzq.reset()
    MotDer.reset()
    MotIzq.on_for_degrees(SpeedPercent(-25), 180, brake=True, block=False)
    MotDer.on_for_degrees(SpeedPercent(25),  180, brake=True, block=False)
    while MotIzq.is_running or MotDer.is_running:
        sleep(0.02)


# --- Loop principal ---
try:
    while True:
        if estado == SIGUIENDO_LINEA:
            if color.color == 5:
                MotIzq.off()
                MotDer.off()
                break
            elif toque.is_pressed:
                MotIzq.on(SpeedPercent(-20),1)    # retrocede 1 segundo
                MotDer.on(SpeedPercent(-20),1)
                girar90Izquierda()                   # gira 90° a la izquierda
                tiempo_rodeo = time()
                estado = RODEANDO
            else:
                seguirLinea()

        elif estado == RODEANDO:
            rodearObstaculo()
            if (time() - tiempo_rodeo) > TIEMPO_MIN and en_linea():
                MotIzq.off()
                MotDer.off()
                estado = SIGUIENDO_LINEA

        sleep(0.05)
finally:
    MotIzq.off()
    MotDer.off()


**Video de ejecución:**

[PENDIENTE: <a href="URL_YOUTUBE_MISION_1" target="_blank">Misión 1 — Bug</a>]

## Misión 2: Superar el laberinto (algoritmo Maze)

**Objetivo:** trasladar el robot EV3 de forma autónoma desde la entrada P1 hasta la salida P2 de un laberinto de dimensiones mínimas 6L × 2L (L = 1.5–2.0 veces el largo del EV3), con entrada y salida claramente marcadas, resolviendo al menos una vez cada situación presentada (pasillo recto, callejón sin salida, intersección).

**Descripción de la solución:** se implementó un seguimiento de pared derecha (regla de la mano derecha). El sensor infrarrojo, orientado al frente, detecta cuándo hay una pared bloqueando el avance; mientras no la detecta, el robot avanza en línea recta. Al llegar a una pared frontal, el robot evalúa con el sensor ultrasónico (orientado a la derecha) si hay pared a ese lado: si la hay, gira 90° a la izquierda; si no la hay (hay una abertura), gira 90° a la derecha y continúa avanzando. Al estar el laberinto organizado en celdas de longitud L, las decisiones de giro coinciden con los puntos donde el robot topa con una pared frontal, cubriendo así pasillos rectos, callejones sin salida (pared al frente y a la derecha → gira izquierda) e intersecciones con abertura a la derecha (gira derecha).

**Nota:** en la versión actual del código el bucle principal (`while True`) no tiene una condición de término: falta definir cómo se marca/detecta la llegada a P2 (p. ej. un color distintivo con un sensor adicional, o la ausencia de pared en un tramo más largo que el resto del laberinto) y agregar esa condición antes de grabar el video final; de lo contrario el robot no se detiene solo al llegar a la salida.

**Pseudocódigo:**

```
inicializar motores, sensor infrarrojo (frontal), sensor ultrasónico (derecho)

repetir:
    avanzar mientras no haya pared al frente (infrarrojo)
    detener motores
    si hay pared a la derecha (ultrasónico < UMBRAL_US):
        girar 90° a la izquierda
    si_no:
        girar 90° a la derecha
    [PENDIENTE: si se detectó la salida P2, detener motores y terminar]
hasta llegar a la salida
```

**Montaje del laberinto:**

[PENDIENTE: foto/diagrama del laberinto con dimensiones (6L × 2L), entrada y salida marcadas. Agregar la imagen a `img/` y referenciarla aquí, p. ej. `![](img/laberinto_montaje.png)`]

**Programa:**


In [ ]:
from time import sleep
from ev3dev2.motor import LargeMotor, OUTPUT_B, OUTPUT_C, SpeedPercent
from ev3dev2.sensor.lego import InfraredSensor, UltrasonicSensor

# --- Hardware ---
MotIzq = LargeMotor(OUTPUT_B)
MotDer = LargeMotor(OUTPUT_C)
infra  = InfraredSensor('in4')
usonic = UltrasonicSensor('in3')

infra.mode  = 'IR-PROX'
usonic.mode = 'US-DIST-CM'

# --- Parámetros ---
VEL_BASE    = 20
GRADOS_GIRO = 180
UMBRAL_IR   = 6    # proximity — pared al frente si < este valor
UMBRAL_US   = 20   # cm — pared a la derecha si < este valor


def hay_pared_frente():
    return infra.proximity < UMBRAL_IR


def hay_pared_derecha():
    return usonic.distance_centimeters < UMBRAL_US


def avanzar():
    while not hay_pared_frente():
        MotIzq.on(SpeedPercent(VEL_BASE))
        MotDer.on(SpeedPercent(VEL_BASE))
        sleep(0.05)
    MotIzq.off()
    MotDer.off()


def girar90Derecha():
    MotIzq.reset()
    MotDer.reset()
    MotIzq.on_for_degrees(SpeedPercent(25),  GRADOS_GIRO, brake=True, block=False)
    MotDer.on_for_degrees(SpeedPercent(-25), GRADOS_GIRO, brake=True, block=False)
    while MotIzq.is_running or MotDer.is_running:
        sleep(0.02)


def girar90Izquierda():
    MotIzq.reset()
    MotDer.reset()
    MotIzq.on_for_degrees(SpeedPercent(-25), GRADOS_GIRO, brake=True, block=False)
    MotDer.on_for_degrees(SpeedPercent(25),  GRADOS_GIRO, brake=True, block=False)
    while MotIzq.is_running or MotDer.is_running:
        sleep(0.02)


# --- Loop principal ---
try:
    while True:
        avanzar()
        if hay_pared_derecha():
            girar90Izquierda()
        else:
            girar90Derecha()
finally:
    MotIzq.off()
    MotDer.off()


**Video de ejecución:**

[PENDIENTE: <a href="URL_YOUTUBE_MISION_2" target="_blank">Misión 2 — Laberinto</a>]

## Videos de ejecución

- [PENDIENTE] <a href="URL_YOUTUBE_MISION_1" target="_blank">Misión 1 — Bug</a>
- [PENDIENTE] <a href="URL_YOUTUBE_MISION_2" target="_blank">Misión 2 — Laberinto</a>

## Experimentación y dificultades

### 1. Duración del retroceso en `Bug.py`

Al revisar el comportamiento de evasión se detectó que la llamada `MotIzq.on(SpeedPercent(-20), 1)` (y su equivalente en `MotDer`) no retrocede al robot durante 1 segundo, como indica el comentario del código. El método `on()` de `ev3dev2` tiene la firma `on(speed, brake=True, block=False)`: no recibe una duración, por lo que ese `1` se interpreta como el argumento `brake`. El retroceso real dura solo lo que tarda en ejecutarse la siguiente línea (`girar90Izquierda()`, que llama `MotIzq.reset()` y detiene el motor casi de inmediato). [PENDIENTE: documentar la corrección aplicada — p. ej. usar `on_for_seconds(SpeedPercent(-20), 1, block=True)` — y el efecto observado en el comportamiento del robot al chocar contra el obstáculo, antes y después de la corrección.]

### 2. Detección de la salida del laberinto

El bucle principal de `laberinto.py` no tiene condición de término. [PENDIENTE: documentar la solución elegida para detectar P2 y las pruebas realizadas para validarla.]

### 3. Calibración de sensores

[PENDIENTE: completar con los resultados obtenidos con `calibrar_color.py` (valor de color de la línea y de la meta) y `calibrar_sensores.py` (umbrales `UMBRAL_IR` y `UMBRAL_US`, y `DIST_CERCA`/`DIST_LEJOS` usados en `Bug.py`), de forma similar a los cálculos documentados en el Laboratorio 1.]

### Conclusión experimental

[PENDIENTE: síntesis de las dificultades encontradas y cómo se resolvieron, siguiendo el mismo formato que la sección de dificultades del Laboratorio 1.]
